# Semantic-Encoder Selection — Turkish Retrieval Benchmark (A100)

**Project:** *Morphology-Aware Contrastive Fine-Tuning for Turkish Retrieval* (inzva AI Projects #10).
The dual encoder has a **semantic** encoder (LoRA fine-tuned) and a **morphological** one. This notebook
ranks candidates for the **semantic** encoder by retrieval quality on 4 Turkish-BEIR datasets. Morphology
is out of scope here (it's the other encoder's job).

**10 candidates** — 8 finished embedders (run zero-shot) + 2 raw MLM foundation models (`TabiBERT`,
`TURKCELL-roberta`) which have no pooling head, get mean-pooled, and are expected to score low: they're
*bases to fine-tune*, not finished retrievers, and are labeled separately.

**4 datasets** (all `trmteb` org, BEIR format): `scifact-tr`, `nfcorpus-tr`, `arguana-tr`, `fiqa-tr`.
**Metric:** nDCG@10 (primary) + Recall@10/100, MRR@10.

> **Runtime:** on an **A100** the full run (incl. fiqa's 57K corpus) is roughly **1–2 h**. Results are cached
> per-model under `results_local/`, so an interrupted run resumes. This is a focused retrieval eval — no
> morphological probe, no gated models.


## 0 · Setup

Run on a **fresh A100 runtime** (Runtime → Change runtime type → A100). Then Run all.

In [ ]:
# transformers PINNED <5: Colab's default pull is v5, which meta-device-inits models and corrupts the
# GTE remote code's non-persistent buffers (TurkEmbed4STS/Retrieval, GTE-mult-base) -> device asserts.
# 4.56+ still covers everything here. Keep hf_xet INSTALLED (some repos are xet-only) but disable its
# use below -- its transfer path hangs/403s on GCP/Colab.
%pip install -q -U "transformers>=4.56,<5" sentence-transformers datasets pandas matplotlib "huggingface_hub[hf_xet]"

In [ ]:
import gc, io, json, math, os, re, shutil, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # disable xet transfer (keeps package); must precede HF imports
import numpy as np
import pandas as pd
import requests
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_url, hf_hub_download
from huggingface_hub.constants import HF_HUB_CACHE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = Path("results_local"); CACHE_DIR.mkdir(exist_ok=True)
FIQA_CORPUS_CAP = None  # A100 handles full fiqa (57K) fine. Set to e.g. 12000 only if time-constrained.

def slug(s): return re.sub(r"[^a-zA-Z0-9]+", "_", s)
print(f"device={DEVICE}  torch={torch.__version__}")
if DEVICE != "cuda":
    print("WARNING: no GPU detected — switch the runtime to A100 (Runtime → Change runtime type).")

### Hugging Face login — **strongly recommended**

None of these 10 models are gated, so login isn't needed for *access*. But an earlier anonymous Colab run
had **~74% of model downloads rejected with `403`** — HF aggressively rate-limits unauthenticated requests
from shared Colab IPs. Logging in (a free read token, ideally saved as the Colab secret `HF_TOKEN`) makes the
run reliable. Uncomment `notebook_login()` if you haven't set the secret.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
from huggingface_hub import whoami
try:
    HF_LOGGED_IN = whoami() is not None
except Exception:
    HF_LOGGED_IN = False
print("HF login:", HF_LOGGED_IN, "" if HF_LOGGED_IN else "-> expect rate-limit 403s; add an HF_TOKEN secret")

## 1 · Candidates (semantic encoders)

`finished` = retrieval/STS-tuned embedder (zero-shot). `raw` = MLM foundation model, mean-pooled zero-shot
(expected low; a base to fine-tune). Query/passage prompts are the models' trained prefixes — omitting them
silently sandbags a model, so they're stored per-model here.

In [ ]:
CANDIDATES = [
    dict(id="intfloat/multilingual-e5-large", short="mE5-large", role="finished",
         query_prompt="query: ", doc_prompt="passage: ", trust_remote_code=False),
    dict(id="newmindai/TurkEmbed4STS", short="TurkEmbed4STS", role="finished",
         query_prompt="", doc_prompt="", trust_remote_code=True),
    dict(id="newmindai/TurkEmbed4Retrieval", short="TurkEmbed4Retrieval", role="finished",
         query_prompt="", doc_prompt="", trust_remote_code=True),
    dict(id="ytu-ce-cosmos/turkish-e5-large", short="turkish-e5-large", role="finished",
         query_prompt="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ",
         doc_prompt="", trust_remote_code=False),
    dict(id="BAAI/bge-m3", short="BGE-M3", role="finished",
         query_prompt="", doc_prompt="", trust_remote_code=False),
    dict(id="Alibaba-NLP/gte-multilingual-base", short="GTE-mult-base", role="finished",
         query_prompt="", doc_prompt="", trust_remote_code=True),
    dict(id="newmindai/Mursit-Base-TR-Retrieval", short="Mursit-Base", role="finished",
         query_prompt="", doc_prompt="", trust_remote_code=False),
    dict(id="trmteb/turkish-embedding-model", short="trmteb-pretrain", role="finished",
         query_prompt="", doc_prompt="", trust_remote_code=False),
    dict(id="boun-tabilab/TabiBERT", short="TabiBERT", role="raw",
         query_prompt="", doc_prompt="", trust_remote_code=False),
    dict(id="TURKCELL/roberta-base-turkish-uncased", short="TURKCELL-roberta", role="raw",
         query_prompt="", doc_prompt="", trust_remote_code=False),
]
print(f"{len(CANDIDATES)} candidates "
      f"({sum(c['role']=='finished' for c in CANDIDATES)} finished / {sum(c['role']=='raw' for c in CANDIDATES)} raw)")

In [ ]:
from sentence_transformers import SentenceTransformer

def _sanity_check(model, cand):
    v = model.encode(["kontrol cümlesi"], convert_to_numpy=True)
    if not np.isfinite(v).all():
        raise RuntimeError(f"{cand['id']}: NaN/inf embeddings — model loaded incorrectly")

def _clear_hf_cache(repo_id):
    d = Path(HF_HUB_CACHE) / f"models--{repo_id.replace('/', '--')}"
    if d.exists():
        shutil.rmtree(d, ignore_errors=True)

def load_model(cand, attempts=3):
    """Retry on any failure (Colab 403s are often transient rate-limits) and clear this repo's cache
    before each retry (a partial download poisons the next attempt with a confusing error)."""
    kwargs = {}
    if DEVICE == "cuda":
        # bf16, never fp16: bf16 has fp32's exponent range (no overflow) and is native on A100.
        kwargs["model_kwargs"] = {"torch_dtype": torch.bfloat16}
    last_err = None
    for i in range(attempts):
        try:
            model = SentenceTransformer(cand["id"], device=DEVICE,
                                        trust_remote_code=cand["trust_remote_code"], **kwargs)
            model.max_seq_length = min(getattr(model, "max_seq_length", 512) or 512, 512)
            _sanity_check(model, cand)
            return model
        except Exception as e:
            last_err = e
            _clear_hf_cache(cand["id"])
            if i < attempts - 1:
                print(f"    [retry {i+1}/{attempts-1}] {cand['id']}: {type(e).__name__}: {str(e)[:150]}")
                time.sleep(2 ** i)
    raise last_err

def encode(model, texts, prompt="", batch_size=128):
    if prompt:
        texts = [prompt + t for t in texts]
    return model.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=True)

def free(model):
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 2 · Datasets (trmteb Turkish-BEIR)

All 4 are the same `trmteb` layout (`corpus`/`queries` configs + qrels under `data/test`). Parquet is fetched
with a resilient helper: `hf_hub_download` first (authenticated, cached, HF's own retry logic), plain `requests`
as fallback, retried with backoff — this routes around the Colab/GCP xet-bridge flakiness.

In [ ]:
BEIR_DATASETS = {
    "scifact-tr": "trmteb/scifact-tr", "nfcorpus-tr": "trmteb/nfcorpus-tr",
    "arguana-tr": "trmteb/arguana-tr", "fiqa-tr": "trmteb/fiqa-tr",
}
EVAL_DATASETS = list(BEIR_DATASETS)
_hf_api = HfApi(); _pq_cache = {}

def _list_parquet(repo):
    if repo not in _pq_cache:
        _pq_cache[repo] = [f for f in _hf_api.list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet")]
    return _pq_cache[repo]

def _read(repo, folder, split=None, attempts=4):
    term = split or folder
    paths = sorted(f for f in _list_parquet(repo) if f.startswith(f"{folder}/") and term in f)
    frames = []
    for p in paths:
        last = None
        for i in range(attempts):
            try:
                frames.append(pd.read_parquet(hf_hub_download(repo, p, repo_type="dataset"))); break
            except Exception as e1:
                last = e1
                try:
                    r = requests.get(hf_hub_url(repo, p, repo_type="dataset"), timeout=180); r.raise_for_status()
                    frames.append(pd.read_parquet(io.BytesIO(r.content))); break
                except Exception as e2:
                    last = e2
            if i < attempts - 1:
                print(f"    [retry {i+1}/{attempts-1}] {repo}/{p}: {type(last).__name__}: {str(last)[:110]}")
                time.sleep(2 ** i)
        else:
            raise last
    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]

def _doc_text(row):
    title = (row.get("title") or "").strip(); text = (row.get("text") or "").strip()
    return f"{title} {text}".strip() if title else text

def load_beir_dataset(name):
    repo = BEIR_DATASETS[name]
    corpus = {str(r["_id"]): _doc_text(r) for r in _read(repo, "corpus").to_dict("records")}
    queries = {str(r["_id"]): r["text"] for r in _read(repo, "queries").to_dict("records")}
    qrels = {}
    for r in _read(repo, "data", "test").to_dict("records"):
        qrels.setdefault(str(r["query-id"]), {})[str(r["corpus-id"])] = float(r["score"])
    queries = {qid: q for qid, q in queries.items() if qid in qrels}  # BEIR convention
    if FIQA_CORPUS_CAP and name == "fiqa-tr" and len(corpus) > FIQA_CORPUS_CAP:
        gold = {d for rels in qrels.values() for d in rels}  # keep every relevant doc
        filler = [d for d in corpus if d not in gold][:FIQA_CORPUS_CAP - len(gold)]
        corpus = {d: corpus[d] for d in list(gold) + filler}
    return corpus, queries, qrels

DATA = {name: load_beir_dataset(name) for name in EVAL_DATASETS}
for name, (c, q, r) in DATA.items():
    print(f"{name}: corpus={len(c)}  queries={len(q)}  qrels={len(r)}")

## 3 · Retrieval metrics

In [ ]:
def dcg(gains):
    return sum(g / math.log2(i + 2) for i, g in enumerate(gains))

def evaluate_run(qrels, run, k_ndcg=10, k_mrr=10, ks_recall=(10, 100)):
    ndcgs, mrrs = [], []; recalls = {k: [] for k in ks_recall}
    for qid, ranked in run.items():
        rel = qrels.get(qid, {})
        if not rel: continue
        ideal = dcg(sorted(rel.values(), reverse=True)[:k_ndcg])
        ndcgs.append(dcg([rel.get(d, 0.0) for d in ranked[:k_ndcg]]) / ideal if ideal > 0 else 0.0)
        mrrs.append(next((1.0 / (i + 1) for i, d in enumerate(ranked[:k_mrr]) if rel.get(d, 0) > 0), 0.0))
        n_rel = sum(1 for v in rel.values() if v > 0)
        for k in ks_recall:
            recalls[k].append(sum(1 for d in ranked[:k] if rel.get(d, 0) > 0) / n_rel if n_rel else 0.0)
    out = {"nDCG@10": np.mean(ndcgs), "MRR@10": np.mean(mrrs)}
    out.update({f"Recall@{k}": np.mean(recalls[k]) for k in ks_recall})
    return {m: round(float(v) * 100, 2) for m, v in out.items()}

def search(query_emb, doc_emb, doc_ids, query_ids, top_k=100, remove_self=False):
    D = torch.from_numpy(doc_emb).to(DEVICE); Q = torch.from_numpy(query_emb).to(DEVICE)
    run = {}
    for start in range(0, len(Q), 256):
        chunk = Q[start:start + 256]
        k = min(top_k + (1 if remove_self else 0), len(doc_ids))
        top = torch.topk(chunk @ D.T, k=k, dim=1).indices.cpu().numpy()
        for row, qi in zip(top, range(start, start + len(chunk))):
            qid = query_ids[qi]
            ranked = [doc_ids[j] for j in row]
            if remove_self:
                ranked = [d for d in ranked if d != qid]
            run[qid] = ranked[:top_k]
    del D, Q
    return run

## 4 · Run — encode, retrieve, score (per-model, cached & resumable)

In [ ]:
rows = []
for cand in CANDIDATES:
    res_path = CACHE_DIR / f"retr_{slug(cand['id'])}.json"
    if res_path.exists():
        rows += json.loads(res_path.read_text()); print(f"[cache] {cand['short']}"); continue
    print(f"\n=== {cand['id']} ({cand['role']}) ===")
    try:
        model = load_model(cand)
    except Exception as e:
        print(f"  [skip] could not load: {type(e).__name__}: {str(e)[:160]}"); continue
    try:
        model_rows = []
        for name in EVAL_DATASETS:
            corpus, queries, qrels = DATA[name]
            doc_ids, query_ids = list(corpus), list(queries)
            t0 = time.time()
            doc_emb = encode(model, [corpus[d] for d in doc_ids], cand["doc_prompt"])
            query_emb = encode(model, [queries[q] for q in query_ids], cand["query_prompt"])
            run = search(query_emb, doc_emb, doc_ids, query_ids, remove_self=(name == "arguana-tr"))
            metrics = evaluate_run(qrels, run); dt = time.time() - t0
            model_rows.append(dict(model=cand["short"], role=cand["role"], dataset=name, **metrics,
                                   encode_sec=round(dt, 1)))
            print(f"  {name}: {metrics}  ({dt:.0f}s)")
        res_path.write_text(json.dumps(model_rows)); rows += model_rows
    except Exception as e:
        print(f"  [skip] error evaluating: {type(e).__name__}: {str(e)[:160]}")
    finally:
        free(model)

retr_df = pd.DataFrame(rows)
retr_df

In [ ]:
pivot = retr_df.pivot_table(index=["model", "role"], columns="dataset", values="nDCG@10")
pivot["mean"] = pivot.mean(axis=1)
pivot = pivot.sort_values("mean", ascending=False).round(2)
pivot.to_csv(CACHE_DIR / "semantic_retrieval.csv")
display(pivot)

In [ ]:
d = pivot["mean"].sort_values()
colors = ["#9ca3af" if role == "raw" else "#2563eb" for _, role in d.index]
fig, ax = plt.subplots(figsize=(8, 0.5 * len(d) + 1.2))
bars = ax.barh([m for m, _ in d.index], d.values, color=colors, height=0.62)
ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
ax.set_title("Semantic-encoder candidates — mean nDCG@10 across 4 trmteb datasets\n"
             "(gray = raw MLM, zero-shot floor / base-to-fine-tune)", fontsize=11, loc="left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e5e7eb", linewidth=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(CACHE_DIR / "semantic_retrieval.png", dpi=140); plt.show()

## 5 · Reading the results

- **Finished embedders** are directly comparable — the top of this table is the strongest semantic-encoder
  base to LoRA-fine-tune. Sanity anchors from prior runs: `turkish-e5-large` and `mE5-large` should land near
  the top (~mid-40s mean nDCG@10); a finished model scoring < 20 signals a **prompt/pooling bug**, not a bad
  model — check its registry prompts first.
- **Raw models** (`TabiBERT`, `TURKCELL-roberta`, gray bars) have no embedding training and score near zero on
  retrieval — expected, not a bug. Their score measures *how much lift fine-tuning must provide*, not final
  quality; don't rule them out as bases on this number alone.
- A **morphological tester** (§6 below) is included only as a *diagnostic* — it shows how much suffix signal
  each base already carries, but it is **not** a selection criterion here (morphology is the other encoder's
  job). Base selection is on retrieval alone.

**Decision:** pick the top finished embedder as the semantic base; keep 2–3 others as report baselines.

## 6 · Morphological tester (2 examples per suffix category)

Same design as the project's pilot table: each item has an **anchor**, a **positive** (a true paraphrase,
different words) and a **distractor** (same root, but a suffix flips the meaning). For every model we show
**sim(anchor, positive)** and **sim(anchor, distractor)** — a model only *passes* an item when the positive
scores higher than the distractor. Anchors use each model's query prompt, candidates its doc prompt (exactly
how an agent-memory lookup runs).

**2 examples × 5 categories** — *hal eki* (case/direction), *olumsuzluk* (negation), *yeterlilik* (ability),
*ettirgen* (causative), *zaman* (tense) = 10 items. This is a **diagnostic**, not a selection criterion:
morphology is the other encoder's job, so a semantic encoder isn't expected to ace it — the value is seeing
*how much* suffix signal each base already carries (and which categories collapse for everyone).

In [ ]:
MORPH_EXAMPLES = [
    # ---- hal eki (case / direction) — same verb, dative<->ablative flip
    dict(category="hal eki", anchor="Son dersten sonra hemen okuldan eve geldim.",
         distractor="Son dersten sonra okula geldim.", positive="Dersler bitince doğruca eve döndüm."),
    dict(category="hal eki", anchor="Geçen ay ailemle birlikte şehre taşındım.",
         distractor="Geçen ay ailemle birlikte şehirden taşındım.", positive="Geçen ay yeni bir kente yerleştim."),
    # ---- olumsuzluk (negation)
    dict(category="olumsuzluk", anchor="Dün sabahki proje toplantısına katıldım.",
         distractor="Dün sabahki proje toplantısına katılmadım.", positive="Dünkü görüşmede ben de vardım."),
    dict(category="olumsuzluk", anchor="Sözleşmeyi dün imzaladım.",
         distractor="Sözleşmeyi dün imzalamadım.", positive="Anlaşma evrakını dün onaylayıp mühürledim."),
    # ---- yeterlilik (ability, negative)
    dict(category="yeterlilik", anchor="Bu ayki elektrik faturasını dün ödedim.",
         distractor="Bu ayki elektrik faturasını dün ödeyemedim.", positive="Elektrik borcumu dün akşam tamamen kapattım."),
    dict(category="yeterlilik", anchor="Sunumu zamanında yetiştirebildim.",
         distractor="Sunumu zamanında yetiştiremedim.", positive="Sunum dosyam tam vaktinde hazırdı."),
    # ---- ettirgen (causative)
    dict(category="ettirgen", anchor="Salondaki kapıyı gelir gelmez açtım.",
         distractor="Salondaki kapıyı gelir gelmez açtırdım.", positive="Gelir gelmez salonun girişini kendim araladım."),
    dict(category="ettirgen", anchor="Arabayı yıkadım.",
         distractor="Arabayı yıkattım.", positive="Aracımı kendim temizledim."),
    # ---- zaman (tense)
    dict(category="zaman", anchor="Siparişim öğlen saatlerinde kargoya verildi.",
         distractor="Siparişim öğlen saatlerinde kargoya verilecek.", positive="Paketim öğlen civarında yola çıktı."),
    dict(category="zaman", anchor="Faturayı ödedim.",
         distractor="Faturayı ödeyeceğim.", positive="Ödemeyi tamamladım, borç kapandı."),
]

# Reloads each model (from HF cache in-session) to score the 10 items; own cache -> resumable,
# and independent of the retrieval cache above.
morph_rows = []
for cand in CANDIDATES:
    mp = CACHE_DIR / f"morph_{slug(cand['id'])}.json"
    if mp.exists():
        morph_rows += json.loads(mp.read_text()); print(f"[cache] {cand['short']}"); continue
    print(f"=== {cand['id']} ({cand['role']}) ===")
    try:
        model = load_model(cand)
    except Exception as e:
        print(f"  [skip] {type(e).__name__}: {str(e)[:120]}"); continue
    try:
        A = encode(model, [t["anchor"] for t in MORPH_EXAMPLES], cand["query_prompt"]).astype(np.float32)
        P = encode(model, [t["positive"] for t in MORPH_EXAMPLES], cand["doc_prompt"]).astype(np.float32)
        D = encode(model, [t["distractor"] for t in MORPH_EXAMPLES], cand["doc_prompt"]).astype(np.float32)
        sp, sd = (A * P).sum(1), (A * D).sum(1)
        recs = [dict(model=cand["short"], role=cand["role"], no=i + 1, category=t["category"],
                     anchor=t["anchor"], positive=t["positive"], distractor=t["distractor"],
                     sim_positive=round(float(sp[i]), 4), sim_distractor=round(float(sd[i]), 4),
                     margin=round(float(sp[i] - sd[i]), 4), passed=bool(sp[i] > sd[i]))
                for i, t in enumerate(MORPH_EXAMPLES)]
        mp.write_text(json.dumps(recs, ensure_ascii=False)); morph_rows += recs
    except Exception as e:
        print(f"  [skip] {type(e).__name__}: {str(e)[:120]}")
    finally:
        free(model)

morph_df = pd.DataFrame(morph_rows)
print(f"\n{morph_df['model'].nunique() if not morph_df.empty else 0} models x {len(MORPH_EXAMPLES)} items")

In [ ]:
if morph_df.empty:
    print("no morphological results produced")
else:
    # The 10 items (2 per category)
    first = morph_df["model"].iloc[0]
    items = (morph_df[morph_df["model"] == first]
             [["no", "category", "anchor", "distractor", "positive"]].reset_index(drop=True))
    print("Items (2 per suffix category):")
    display(items)

    # Deck-style grid: rows = item, cols = model -> "sim_positive / sim_distractor  (✓ positive wins / ✗)"
    def _cell(r):
        return f"{r.sim_positive:.3f} / {r.sim_distractor:.3f} {'✓' if r.passed else '✗'}"
    grid = (morph_df.assign(cell=morph_df.apply(_cell, axis=1))
            .pivot_table(index=["no", "category"], columns="model", values="cell", aggfunc="first"))
    grid = grid[[c["short"] for c in CANDIDATES if c["short"] in grid.columns]]  # registry column order
    print("\nPer item:  sim(anchor, positive) / sim(anchor, distractor)   [✓ = positive scored higher]")
    with pd.option_context("display.max_columns", None, "display.width", None):
        display(grid)

    # Per-model summary — how much suffix signal each base already carries
    summary = (morph_df.groupby(["model", "role"])
               .agg(passed=("passed", "sum"), items=("passed", "count"), mean_margin=("margin", "mean"))
               .reset_index())
    summary["pass_%"] = (summary["passed"] / summary["items"] * 100).round(0)
    summary = summary.sort_values("mean_margin", ascending=False).round(3)
    print("\nPer-model summary (passed = positive beat distractor; margin = mean sim_pos − sim_dist):")
    display(summary[["model", "role", "passed", "items", "pass_%", "mean_margin"]])

In [ ]:
# Deck-style pilot table: grouped "Örnek" header + per-model (dist / pos / sonuç), colored ok/FAIL.
# With many models this gets very wide — set MORPH_TABLE_MODELS to a couple of names for the deck's
# 2-column look, e.g. ["mE5-large", "BGE-M3"]. Default = every evaluated model, in registry order.
MORPH_TABLE_MODELS = [c["short"] for c in CANDIDATES if c["short"] in set(morph_df["model"])]

if morph_df.empty:
    print("run the §6 compute cell first")
else:
    _base = (morph_df[morph_df["model"] == MORPH_TABLE_MODELS[0]].sort_values("no")
             [["anchor", "distractor", "positive", "category"]].reset_index(drop=True))
    _base.columns = pd.MultiIndex.from_tuples(
        [("Örnek", "Anchor"), ("Örnek", "Distractor"), ("Örnek", "Positive"), ("Örnek", "Ek Türü")])
    _blocks = [_base]
    for m in MORPH_TABLE_MODELS:
        s = morph_df[morph_df["model"] == m].sort_values("no").reset_index(drop=True)
        b = pd.DataFrame({(m, "dist"): s["sim_distractor"].values,
                          (m, "pos"): s["sim_positive"].values,
                          (m, "sonuç"): np.where(s["passed"], "ok", "FAIL")})
        b.columns = pd.MultiIndex.from_tuples(b.columns)
        _blocks.append(b)
    pilot = pd.concat(_blocks, axis=1)

    def _color_sonuc(v):
        if v == "FAIL": return "background-color:#f8d7da;color:#b02a37;font-weight:600;text-align:center"
        if v == "ok":   return "background-color:#d1e7dd;color:#0f5132;font-weight:600;text-align:center"
        return ""

    _sonuc = [(m, "sonuç") for m in MORPH_TABLE_MODELS]
    _num = {(m, c): "{:.4f}" for m in MORPH_TABLE_MODELS for c in ("dist", "pos")}
    sty = pilot.style
    sty = sty.map(_color_sonuc, subset=_sonuc) if hasattr(pilot.style, "map") else sty.applymap(_color_sonuc, subset=_sonuc)
    sty = (sty.format(_num)
           .set_properties(subset=[("Örnek", c) for c in ("Anchor", "Distractor", "Positive", "Ek Türü")],
                           **{"text-align": "left", "white-space": "normal", "max-width": "260px"})
           .set_table_styles([
               {"selector": "th", "props": [("font-size", "12px"), ("text-align", "center"), ("padding", "4px 8px")]},
               {"selector": "td", "props": [("font-size", "12px"), ("padding", "4px 8px"), ("text-align", "center")]},
               {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#f4f4f4")]},
           ])
           .set_caption("Morphological pilot — dist = sim(anchor, distractor), pos = sim(anchor, positive); "
                        "sonuç = ok if the positive scored higher, else FAIL"))
    try:
        html = sty.to_html()
        with open(CACHE_DIR / "morph_pilot_table.html", "w") as f:
            f.write(html)
    except Exception:
        pass
    sty  # renders inline in Colab